# broadcasting-rules — worked example 1: Predict the broadcast shape of three shape tuples

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `broadcasting-rules`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Concept

Broadcasting **right-aligns** shapes, left-padding the shorter with 1s. For each aligned axis: equal keeps, a `1` stretches to the other, and two unequal non-1 sizes are **incompatible**. The rule is associative, so three tensors broadcast by folding pairwise left-to-right.

## Worked solution

We want the final shape of broadcasting `(4, 1, 6)`, `(3, 6)`, and `(1,)` together.

**Step 1 — fold the first two.** Right-align `(4, 1, 6)` and `(3, 6)`. The shorter one is left-padded: `(3, 6)` becomes `(1, 3, 6)`. Compare axis-by-axis: `4 vs 1 -> 4`, `1 vs 3 -> 3`, `6 vs 6 -> 6`. Result so far: `(4, 3, 6)`. This works because every mismatch involves a `1`, which legally stretches.

**Step 2 — fold in the third.** Right-align `(4, 3, 6)` with `(1,)`. Left-pad `(1,)` to `(1, 1, 1)`. Compare: `4 vs 1 -> 4`, `3 vs 1 -> 3`, `6 vs 1 -> 6`. Result: `(4, 3, 6)`. A scalar-like `(1,)` stretches against everything.

**Step 3 — why fold pairwise?** Broadcasting is associative and commutative on shapes, so reducing the list two-at-a-time gives the same answer as aligning all three at once. If any pairwise step hits two unequal non-1 sizes, the whole thing is incompatible and we raise.

The implementation reuses a single `_pair` helper and `functools.reduce` to fold the list.

In [ ]:
from functools import reduce

def broadcast_shapes(*shapes):
    def _pair(a, b):
        a, b = list(a), list(b)
        n = max(len(a), len(b))
        a = [1] * (n - len(a)) + a
        b = [1] * (n - len(b)) + b
        out = []
        for ai, bi in zip(a, b):
            if ai == bi or bi == 1:
                out.append(ai)
            elif ai == 1:
                out.append(bi)
            else:
                raise ValueError(f'incompatible axes: {ai} vs {bi}')
        return tuple(out)
    return reduce(_pair, shapes, (1,))

result = broadcast_shapes((4, 1, 6), (3, 6), (1,))
print('broadcast shape =', result)
# Cross-check against torch's own broadcaster.
ref = tuple(t.broadcast_shapes((4, 1, 6), (3, 6), (1,)))
print('torch reference =', ref, '| match:', result == ref)